# 2026-11-14 Training pairs for threshold learning

**What this does:** extracts XFeat descriptors for 200 training + 64 development COCO val2017 images (3 synthetic warps each) that are disjoint from the 192 calibration and 64 validation images. It uses the original, unmodified `features.py`.

**Before running:** upload this whole folder to `MyDrive/Colab Notebooks/2026-11-14_learned_thresholds/`.

**Runtime:** GPU (T4) is fine and faster; CPU also works. The notebook downloads `val2017.zip` (about 1 GB) from the COCO site, so nothing large needs to be uploaded.

**After running:** download the new `train_pairs_YYYYMMDD_HHMMSS/` folder from Drive and put its three files into `CSE527/data/2026-11-14_train_pairs/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
EXP=Path('/content/drive/MyDrive/Colab Notebooks/2026-11-14_learned_thresholds')
for p in [EXP/'make_train_pairs.py',EXP/'inputs'/'orig_code'/'features.py',EXP/'inputs'/'coco_manifest_2026-10-14.json']:
    assert p.exists(),p
print('inputs found')

In [ ]:
import subprocess,sys,importlib.util
for module,package in {'cv2':'opencv-python-headless','torch':'torch'}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',package],check=True)
subprocess.run(['rm','-rf','/content/run'],check=True)
subprocess.run(['cp','-r',str(EXP),'/content/run'],check=True)
if not Path('/content/val2017.zip').exists():
    subprocess.run(['wget','-q','http://images.cocodataset.org/zips/val2017.zip','-O','/content/val2017.zip'],check=True)
print('zip MB',Path('/content/val2017.zip').stat().st_size/2**20)

In [ ]:
import time
start=time.time()
r=subprocess.run([sys.executable,'/content/run/make_train_pairs.py','/content/run/inputs/orig_code',
                  '/content/run/inputs/coco_manifest_2026-10-14.json','/content/val2017.zip','/content/work','/content/out'],
                 capture_output=True,text=True)
print(r.stdout[-3000:]); print(r.stderr[-3000:]); print('minutes',(time.time()-start)/60)
assert r.returncode==0

In [ ]:
# Save to Drive (new folder; nothing existing is overwritten)
out=EXP/f'train_pairs_{time.strftime("%Y%m%d_%H%M%S")}'
subprocess.run(['cp','-r','/content/out',str(out)],check=True)
print('saved to',out); print(subprocess.run(['ls','-la',str(out)],capture_output=True,text=True).stdout)